## cleaning

In [1]:
# Prima fotografia delle 9 tabelle: tipi (le date arrivano come testo) e mancanti.
# dfs è il contenitore unico: ogni tabella pulita ci rientra e alla fine viene salvato tutto nel pickle.
import pandas as pd
from pathlib import Path

raw = Path("data/raw") # path sottocartella dei file

files = {
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {} # dizionario per avere sempre un modo comodo per chiamare le tabelle

for name, f in files.items(): 
    df = pd.read_csv(raw / f) 
    dfs[name] = df
    print(f"--- {name} ---")
    print("shape:", df.shape)
    print(df.dtypes)
    print("missing:\n", df.isna().sum()[df.isna().sum() > 0])
    print()

--- orders ---
shape: (99441, 8)
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
missing:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- items ---
shape: (112650, 7)
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
missing:
 Series([], dtype: int64)

--- payments ---
shape: (103886, 5)
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object
missing:


## orders

In [2]:
# Date da testo a datetime: servono per calcolare durate (blocco G) e raggruppare per mese. I mancanti diventano NaT.
orders = dfs["orders"].copy() #copio per pulire il df

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime) # visto prima che era in str lo trasformo datetime
print(orders[date_cols].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [3]:
# order_id unico = chiave primaria (una riga = un ordine).
print(orders["order_id"].is_unique, orders.duplicated().sum()) # unicità del id e duplicati

True 0


In [4]:
# Un mancante non è per forza un errore: si guarda se lo stato dell'ordine lo spiega (cancellato, in viaggio...).
# Quasi tutti sono spiegati e restano com'è. Gli unavailable restano: sono ordini reali (contano negli stati) ma quasi tutti senza item, quindi senza fatturato.
# Anomalie vere: delivered senza data (14 senza approvazione, 2 senza corriere, 8 senza consegna): dato mancante vero, non inventabile.
for c in ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]:
    print(c)
    print(orders.loc[orders[c].isna(), "order_status"].value_counts())
    print()

order_approved_at
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

order_delivered_carrier_date
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

order_delivered_customer_date
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64



In [5]:
# Date impossibili (166 + 23): non si correggono (la data vera non si conosce), si escludono solo dai calcoli di durata
# (filtro valid, blocco G). Le 23 non danno problemi: nessuna metrica usa il tempo corriere -> cliente.
print((orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum()) # impossibile, probabile errore di inserimento escludo per logistica tengo per fatturato
print((orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print((orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum())

166
0
23


In [6]:
dfs["orders"] = orders

## items

In [7]:
items = dfs["items"].copy()
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
print(items["shipping_limit_date"].dtype)

datetime64[us]


In [8]:
# Chiave composta: un ordine ha più item, ogni item è identificato da order_id + order_item_id.
print(items.duplicated(["order_id", "order_item_id"]).sum())
print(items.duplicated().sum())

0
0


In [9]:
# Si distingue "valore impossibile" (<= 0, negativo: nessuno) da "alto ma plausibile" (prezzo massimo 6.735: si tiene).
# 383 trasporti a 0 = spedizione gratuita. Media molto sopra la mediana: coda lunga, nelle analisi meglio le mediane.
print(items[["price", "freight_value"]].describe())
print((items["price"] <= 0).sum(), (items["freight_value"] < 0).sum())
print((items["freight_value"] == 0).sum())

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000
0 0
383


In [10]:
# Integrità tra tabelle: ogni item ha un ordine. I 775 ordini senza item (unavailable e canceled) sono normali:
# non hanno fatturato ma restano nei conteggi degli stati.
orders = dfs["orders"]
print(items["order_id"].isin(orders["order_id"]).all())

no_items = ~orders["order_id"].isin(items["order_id"])
print(no_items.sum())
print(orders.loc[no_items, "order_status"].value_counts())

True
775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [11]:
dfs["items"] = items

## payments

In [12]:
# Chiave composta order_id + payment_sequential: un ordine può avere più pagamenti (103.886 righe per 99.440 ordini),
# quindi per contare ordini servirà nunique.
payments = dfs["payments"].copy()
print(payments.dtypes)
print(payments.duplicated(["order_id", "payment_sequential"]).sum())

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object
0


In [13]:
print(payments["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [14]:
print(payments[["payment_value", "payment_installments"]].describe())
print((payments["payment_value"] < 0).sum())
print((payments["payment_value"] == 0).sum())
print((payments["payment_installments"] == 0).sum())

       payment_value  payment_installments
count  103886.000000         103886.000000
mean      154.100380              2.853349
std       217.494064              2.687051
min         0.000000              0.000000
25%        56.790000              1.000000
50%       100.000000              1.000000
75%       171.837500              4.000000
max     13664.080000             24.000000
0
9
2


In [15]:
# Anomalie isolate e irrilevanti, si lasciano: 9 pagamenti a 0 (6 voucher, plausibile; 3 not_defined) e 2 carte con 0 rate.
print(payments.loc[payments["payment_value"] == 0, "payment_type"].value_counts())
print(payments.loc[payments["payment_installments"] == 0, "payment_type"].value_counts())

payment_type
voucher        6
not_defined    3
Name: count, dtype: int64
payment_type
credit_card    2
Name: count, dtype: int64


In [16]:
# Un solo ordine consegnato senza pagamento: irrilevante.
print(payments["order_id"].isin(orders["order_id"]).all())

no_payments = ~orders["order_id"].isin(payments["order_id"])
print(no_payments.sum())
print(orders.loc[no_payments, "order_status"].value_counts())

True
1
order_status
delivered    1
Name: count, dtype: int64


In [17]:
dfs["payments"] = payments

## reviews

In [18]:
reviews = dfs["reviews"].copy()
print(reviews.dtypes)

date_cols = ["review_creation_date", "review_answer_timestamp"]
reviews[date_cols] = reviews[date_cols].apply(pd.to_datetime)
print(reviews[date_cols].dtypes)

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [19]:
# Né review_id né order_id sono chiavi: la cella 28 spiega perché.
print(reviews["review_id"].is_unique)
print(reviews["order_id"].is_unique)
print(reviews.duplicated().sum())

False
False
0


In [20]:
# Voto da 1 a 5 senza valori fuori scala, molto sbilanciato (58% di 5): la media da sola nasconde la polarizzazione.
print(reviews["review_score"].describe())
print(reviews["review_score"].value_counts().sort_index())

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [21]:
# Testi mancanti normali: il voto è obbligatorio, il testo facoltativo. Non si riempiono.
print(reviews["review_comment_title"].isna().mean())
print(reviews["review_comment_message"].isna().mean())

0.8834153027493349
0.5870253164556962


In [22]:
print((reviews["review_answer_timestamp"] < reviews["review_creation_date"]).sum())

0


In [23]:
# 768 ordini senza recensione (quasi tutti delivered e shipped): normale, non tutti recensiscono.
print(reviews["order_id"].isin(orders["order_id"]).all())

no_reviews = ~orders["order_id"].isin(reviews["order_id"])
print(no_reviews.sum())
print(orders.loc[no_reviews, "order_status"].value_counts())

True
768
order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64


In [24]:
# 547 ordini hanno più recensioni e 789 review_id compaiono su ordini diversi: anomalie note del dataset, indipendenti tra loro.
# Implicazione: order_id non è chiave in reviews, prima di ogni join va aggregato (media per ordine). review_id non si usa come chiave.
print(reviews["order_id"].value_counts().value_counts())
print(reviews["review_id"].value_counts().value_counts())

count
1    98126
2      543
3        4
Name: count, dtype: int64
count
1    97621
2      764
3       25
Name: count, dtype: int64


In [25]:
dfs["reviews"] = reviews

## costumers

In [26]:
customers = dfs["customers"].copy()
print(customers.dtypes)

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object


In [27]:
print(customers["customer_id"].is_unique)
print(customers.duplicated().sum())

True
0


In [28]:
# customer_id cambia a ogni ordine, customer_unique_id identifica la persona: 96.096 persone su 99.441 ordini, 2.997 (3,1%) hanno riacquistato.
# Le analisi sui clienti si fanno su customer_unique_id. Il riacquisto basso è il motivo per cui retention e cohort non sono state tenute.
print(customers["customer_unique_id"].nunique())
print(len(customers))
print(customers["customer_unique_id"].value_counts().value_counts())

96096
99441
count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
17        1
9         1
Name: count, dtype: int64


In [29]:
# I CAP sono interi, quindi perdono lo zero iniziale (1003 = 01003): non è un problema, è coerente con geolocation e i join funzionano.
print(customers["customer_zip_code_prefix"].describe())
print(customers["customer_state"].value_counts())
print(customers["customer_city"].nunique())

count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64
4119


In [30]:
print(customers.isna().sum())

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [31]:
# Integrità in entrambe le direzioni: il merge orders-customers non perde né crea righe.
print(customers["customer_id"].isin(orders["customer_id"]).all())
print(orders["customer_id"].isin(customers["customer_id"]).all())

True
True


In [32]:
dfs["customers"] = customers

## products

In [33]:
products = dfs["products"].copy()
print(products.dtypes)

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [34]:
print(products["product_id"].is_unique)
print(products.duplicated().sum())

True
0


In [35]:
print(products.isna().sum())

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [36]:
# I 610 mancanti (categoria, nome, descrizione, foto) sono gli stessi prodotti: scheda mai compilata.
# Non si possono riempire in modo sensato: restano senza categoria e nell'EDA sono fuori dai confronti per categoria.
cols = ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
print(products[cols].isna().sum())
print(products[products["product_category_name"].isna()][cols].head())

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64
    product_category_name  product_name_lenght  product_description_lenght  \
105                   NaN                  NaN                         NaN   
128                   NaN                  NaN                         NaN   
145                   NaN                  NaN                         NaN   
154                   NaN                  NaN                         NaN   
197                   NaN                  NaN                         NaN   

     product_photos_qty  
105                 NaN  
128                 NaN  
145                 NaN  
154                 NaN  
197                 NaN  


In [37]:
# Dimensioni mai <= 0, ma 4 prodotti hanno peso 0 (impossibile): trascurabile, il peso non è usato. Se servisse, va messo a NaN.
dim_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print(products[dim_cols].describe())
print((products[dim_cols] <= 0).sum())

       product_weight_g  product_length_cm  product_height_cm  \
count      32949.000000       32949.000000       32949.000000   
mean        2276.472488          30.815078          16.937661   
std         4282.038731          16.914458          13.637554   
min            0.000000           7.000000           2.000000   
25%          300.000000          18.000000           8.000000   
50%          700.000000          25.000000          13.000000   
75%         1900.000000          38.000000          21.000000   
max        40425.000000         105.000000         105.000000   

       product_width_cm  
count      32949.000000  
mean          23.196728  
std           12.079047  
min            6.000000  
25%           15.000000  
50%           20.000000  
75%           30.000000  
max          118.000000  
product_weight_g     4
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [38]:
# 98,1% con traduzione: il resto sono i 610 senza categoria e 13 prodotti in due categorie senza traduzione
# (pc_gamer e portateis_cozinha_e_preparadores_de_alimentos). category_translation non ha una pulizia propria: viene solo usata.
translation = dfs["category_translation"]
print(products["product_category_name"].isin(translation["product_category_name"]).mean())

0.9810931382962581


In [39]:
items = dfs["items"]
print(items["product_id"].isin(products["product_id"]).all())

True


In [40]:
dfs["products"] = products

## sellers

In [41]:
sellers = dfs["sellers"].copy()
print(sellers.dtypes)

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object


In [42]:
print(sellers["seller_id"].is_unique)
print(sellers.duplicated().sum())

True
0


In [43]:
print(sellers.isna().sum())

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


In [44]:
# Venditori concentrati in SP (1.849 su 3.095): dato reale, spiega i flussi SP -> SP dell'EDA.
print(sellers["seller_state"].nunique())
print(sellers["seller_state"].value_counts())
print(sellers["seller_city"].nunique())

23
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64
611


In [45]:
items = dfs["items"]
print(items["seller_id"].isin(sellers["seller_id"]).all())

True


In [46]:
dfs["sellers"] = sellers

## geoloc

In [47]:
geo = dfs["geolocation"].copy()
print(geo.dtypes)

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


In [48]:
# 1 milione di righe per 19.015 CAP. I 261.831 duplicati non si tolgono: l'EDA riduce comunque a una riga per CAP con la media
# (togliendoli il centro di un CAP si sposta in mediana di 35 metri).
print(geo.duplicated().sum())
print(geo["geolocation_zip_code_prefix"].nunique())
print(len(geo))

261831
19015
1000163


In [49]:
print(geo.isna().sum())

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


In [50]:
print(geo[["geolocation_lat", "geolocation_lng"]].describe())

       geolocation_lat  geolocation_lng
count     1.000163e+06     1.000163e+06
mean     -2.117615e+01    -4.639054e+01
std       5.715866e+00     4.269748e+00
min      -3.660537e+01    -1.014668e+02
25%      -2.360355e+01    -4.857317e+01
50%      -2.291938e+01    -4.663788e+01
75%      -1.997962e+01    -4.376771e+01
max       4.506593e+01     1.211054e+02


In [51]:
# Rettangolo grezzo del Brasile (lat da -34 a +5, lng da -74 a -34): le coordinate fuori sono errori di geocodifica (Europa, Asia...).
# Non intercetta gli errori dentro la scatola (306 CAP con punti sparsi oltre 110 km, 1,6%): limite noto, una mediana per CAP sarebbe più robusta.
fuori_brasile = geo[
    (geo["geolocation_lat"] > 5) | (geo["geolocation_lat"] < -34) |
    (geo["geolocation_lng"] > -34) | (geo["geolocation_lng"] < -74)
]
print(len(fuori_brasile))

42


In [52]:
# Unica eliminazione di righe del cleaning (42): una coordinata impossibile non si può correggere.
geo = geo[
    (geo["geolocation_lat"] <= 5) & (geo["geolocation_lat"] >= -34) &
    (geo["geolocation_lng"] <= -34) & (geo["geolocation_lng"] >= -74)
]
print(len(geo))


1000121


In [53]:
print(fuori_brasile[["geolocation_lat", "geolocation_lng"]].describe())

       geolocation_lat  geolocation_lng
count        42.000000        42.000000
mean         17.753525       -25.553428
std          27.421425        37.923379
min         -36.605374      -101.466766
25%          -3.847972       -32.423525
50%          28.709115       -12.468452
75%          41.556871        -7.026689
max          45.065933       121.105394


In [54]:
dfs["geolocation"] = geo

## pickle

In [55]:
# Salva tutto il dizionario: l'EDA riparte da qui senza rieseguire il cleaning.
import pickle
from pathlib import Path

processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)

with open(processed / "dfs_clean.pkl", "wb") as f:
    pickle.dump(dfs, f)